# Fiaz et al. (2026) 검산 노트북 — Zafar 데이터셋 실측
WalkAI LAB · 2026-09-10 · 김지민

논문 **"Machine Learning Models for Reliable Gait Phase Detection Using Lower-Limb Wearable Sensor Data"**
(Appl. Sci. 16(3):1397, [10.3390/app16031397](https://doi.org/10.3390/app16031397))의 수치를 원시 데이터로 직접 검산한다.

## 검산 항목
1. **윈도우 수 모순** — 논문 자체 수치만으로 성립 (데이터 불필요, 산수)
2. **라벨 기존재·품질** — 원 데이터셋(Zafar, figshare article 11881332)의 이벤트/트리거 열 실측
3. **논문 총 샘플 433,932의 산출 근거** — 보행 모드 샘플 수 대조

## 준비물
- `Dataset.zip` (2.0 GB): https://ndownloader.figshare.com/files/33417746
  (figshare article: *Development and Evaluation of a Low-Cost Data Acquisition System using Heterogeneous Sensors*, v9)
- Python 3.9+ 표준 라이브러리만 사용 (pandas 불필요)

In [ ]:
# 설정 — Dataset.zip 경로를 환경에 맞게 수정
ZIP_PATH = "Dataset.zip"

from zipfile import ZipFile
import csv, io, json, collections

z = ZipFile(ZIP_PATH)
print("zip entries:", len(z.namelist()))

## 검산 ① — 윈도우 수 모순 (데이터 불필요)

논문 §3.1.1: stance 186,468 / swing 125,033 / mid-swing 122,431 **샘플**.
논문 §3.1.2: 윈도우 길이 L=128, stride 64 (50% overlap).
논문 Table 6: "Total Windows" = 위 샘플 수와 동일한 433,932.

저자가 쓴 규칙대로 계산하면:

In [ ]:
samples = 186_468 + 125_033 + 122_431
L, stride = 128, 64
windows_expected = (samples - L) // stride + 1  # 트라이얼 경계 무시한 상한
print(f"총 샘플            : {samples:,}")
print(f"규칙대로 윈도우 수  : {windows_expected:,} (상한)")
print(f"Table 6 표기       : {samples:,}  ← 샘플 수와 동일")
print(f"배율               : {samples / windows_expected:.0f}x")
print()
print("Table 6이 맞으려면 stride=1 (overlap 99.2%) 이어야 하고,")
print("그 경우 인접 윈도우 누수는 본문 서술(50%)보다 훨씬 심각해진다.")
print("→ 어느 쪽이든 방법 서술과 결과 표가 정면 모순 (확정)")

## 데이터 구조 확인

- 피험자 6명: HP112/114/115/116/117 + UP112. **주의: HP117 폴더는 `DAQ_HP116/` 안에 중첩**되어 있다 (배포 실수로 보임).
- 각 피험자: `Raw` / `Post` / `MVC`, 전진(F)/후진(B) 트라이얼, 전부 CSV.
- Post CSV 58열: A~W 오른다리(23채널) + X=Mode, Z~AV 왼다리 + AW=Mode, **AY~BF = Heel Contact / Toe Off 이벤트 + 4자리 Trigger 코드** (앞모드·앞위상·뒤모드·뒤위상, Phase 1=Stance 2=Swing 3=Mid-swing).

In [ ]:
# 헤더 실물 확인
import fnmatch
sample = [n for n in z.namelist() if fnmatch.fnmatch(n, 'DAQ_HP112/HP112_Post/HP112_F_Post/*/*R1_F_Post.csv')][0]
raw = z.read(sample).decode('utf8', errors='replace').replace('\x00','')
header = next(csv.reader(io.StringIO(raw, newline='')))
print(sample, '| cols:', len(header))
for i in (0,3,4,6,18,20,23,50,51,52,53):
    print(f"  [{i:2d}] {header[i]}")

## 검산 ②·③ — 전수 집계

Post CSV 288개를 전부 파싱해 (a) 모드별 샘플 수, (b) 이벤트/트리거 열 커버리지를 센다.
CSV에 NUL 문자·비정상 개행이 섞여 있어 전처리가 필요하다. 실행 시간 약 1~3분.

In [ ]:
MODE_NAMES = {0:'S',1:'SitToStand',2:'LW',3:'SA',4:'BW',5:'RD',6:'SD',7:'SW',8:'RA'}
post = [n for n in z.namelist() if '_Post/' in n and n.endswith('.csv') and 'MVC' not in n]
print('post files:', len(post))

tot = 0
mode_counts = collections.Counter()
event_counts = collections.Counter()
files_with_events = 0
per_subject_walk = collections.Counter()
trigger_samples = []

for n in post:
    subj = 'HP117' if 'DAQ_HP117' in n else n.split('/')[0].replace('DAQ_','')
    raw = z.read(n).decode('utf8', errors='replace').replace('\x00','')
    rows = csv.reader(io.StringIO(raw, newline='')); next(rows)
    ev_here = 0
    for r in rows:
        if len(r) < 24: continue
        tot += 1
        try: m = int(float(r[23])) if r[23].strip() else -1
        except ValueError: m = -1
        mode_counts[m] += 1
        if m in (2,8,5): per_subject_walk[subj] += 1
        if len(r) >= 58:
            for ci, cname in ((50,'R_HC'),(52,'R_TO'),(54,'L_HC'),(56,'L_TO')):
                if r[ci].strip() not in ('','0','0.0'):
                    event_counts[cname] += 1; ev_here += 1
                    if cname=='R_HC' and len(trigger_samples)<8 and r[ci+1].strip().isdigit():
                        trigger_samples.append(r[ci+1].strip())
    if ev_here: files_with_events += 1

print(f"총 샘플(전 모드): {tot:,}")
print({MODE_NAMES.get(k,k): f"{v:,}" for k,v in sorted(mode_counts.items())})

In [ ]:
# 검산 ③: 논문 433,932의 산출 근거 대조
walk = mode_counts[2] + mode_counts[8] + mode_counts[5]
print(f"보행(LW/RA/RD) 샘플 : {walk:,}")
print(f"LW만               : {mode_counts[2]:,}")
print(f"논문 총 샘플       : 433,932")
print(f"→ 보행 전체의 {433_932/walk*100:.0f}% / LW만의 {433_932/mode_counts[2]*100:.0f}%")
print("→ 전 트라이얼·전진만·LW만 어느 자연 부분집합과도 불일치.")
print("  자체 라벨링 후 유효 HS-TO-HS 구간만 남긴 결과로 추정되나 논문에 근거 서술 없음.")
print()
print("피험자별 보행 샘플:", {k: f"{v:,}" for k,v in sorted(per_subject_walk.items())})

In [ ]:
# 검산 ②: 라벨(이벤트) 커버리지
print(f"이벤트 보유 파일   : {files_with_events} / {len(post)} ({files_with_events/len(post)*100:.0f}%)")
print(f"이벤트 수          : {dict(event_counts)}")
strides_expected = walk // 100  # 주기당 ~100샘플(100Hz, ~1s) 가정
print(f"예상 보행주기(대략) : ~{strides_expected:,}")
print(f"R_HC 커버리지      : ~{event_counts['R_HC']/strides_expected*100:.0f}%")
print(f"트리거 코드 예시    : {trigger_samples}")
print("  예: '2321' = Mode2(LW)·Phase3(Mid-swing) → Mode2·Phase1(Stance)")
print("  → 3위상 코딩(1=Stance 2=Swing 3=Mid-swing)이 실데이터에 존재")

## 판정 (2026-09-10 실측 기준)

| 항목 | 판정 | 근거 |
|---|---|---|
| 윈도우 수 모순 | **확정 (오류)** | 논문 내부 수치만으로 성립 — 서술(stride 64) vs Table 6, 64배 차이 |
| 라벨 체계 기존재 | **확정** | 이벤트·트리거 열과 3위상 코드가 실데이터에 존재 |
| 라벨 품질 | **심각하게 불완전** | ~30% 파일에만 이벤트, 주기 커버리지 ~3%, HP112는 0개 |
| 논문 relabeling의 정당성 | **실질 정당** | 이 라벨로는 학습 불가. 단 "라벨이 없다"는 서술은 부정확 — "있으나 불완전"이 사실이고 논문은 이를 미언급 |
| 논문 총 샘플 433,932 | **산출 근거 불명** | 실측 보행 샘플 2.93M의 15% (LW만 대비 32%) |

**시사점**: 논문의 자동 relabeling 파이프라인 자체는 유효한 기여다. 문제는 (a) 데이터 출처
서술의 부정확성, (b) 윈도우 수의 내부 모순, (c) 샘플 산출 과정의 불투명성 — 재현 가능성을
깎는 요소들이다. 5주차 Camargo 실습에서 "논문 수치 검산"을 표준 단계로 넣자는 근거.